# 🚀 TARS — LoRA Fine-Tuning
**Model:** `Qwen2.5-0.5B-Instruct`  
**Yöntem:** LoRA (PEFT) + SFT  
**Veri:** JSONL formatında instruction/input/output çiftleri  

### Adımlar
1. Kurulum
2. Veri yükleme ve formatlama
3. Model + LoRA kurulumu
4. Eğitim
5. Test & sohbet

## 📦 1. Kurulum

In [ ]:
# Gerekli kütüphaneleri kur
!pip install -q transformers peft accelerate datasets bitsandbytes trl
print('✅ Kurulum tamamlandı')

In [ ]:
import torch
import json
import os
from pathlib import Path

# GPU kontrolü
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB')
else:
    print('⚠️  GPU bulunamadı — CPU ile çalışılacak (çok yavaş olabilir)')

## 📂 2. Veri Yükleme

Sol panelden **Files → Upload** ile JSONL dosyalarını yükleyin.  
Beklenen format her satırda:
```json
{"instruction": "...", "input": "...", "output": "..."}
```
Eğer `input` boşsa sorun değil.

In [ ]:
# ── Ayarlar ──────────────────────────────────────────────────────────────
JSONL_DIR   = '/content'          # Dosyaları buraya yükleyin (veya Google Drive bağlayın)
MODEL_NAME  = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT_DIR  = '/content/tars_lora'
MAX_LENGTH  = 512                 # Token limiti (T4 için güvenli)
BATCH_SIZE  = 4                   # T4 16GB için uygun
GRAD_ACCUM  = 4                   # Efektif batch = 4×4 = 16
EPOCHS      = 3
LR          = 2e-4
LORA_R      = 16
LORA_ALPHA  = 32
LORA_DROP   = 0.05
# ─────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

# JSONL dosyalarını oku
raw_data = []
jsonl_files = list(Path(JSONL_DIR).glob('*.jsonl'))

if not jsonl_files:
    print('⚠️  JSONL dosyası bulunamadı!')
    print('   → Sol panelden dosya yükleyin veya JSONL_DIR değişkenini değiştirin.')
    print('   → Şimdilik 10 satırlık örnek veri oluşturuluyor...')
    # Demo veri
    demo = [
        {"instruction": "ADC biriminin roket aviyonik sistemindeki rolü nedir?", "input": "Veri dönüştürme",
         "output": "Analog sensörlerden gelen voltaj değerlerini uçuş bilgisayarının işleyebileceği dijital verilere dönüştürür."},
        {"instruction": "IMU sensörünün uçuş kontrol sistemindeki işlevi nedir?", "input": "",
         "output": "İvme, açısal hız ve manyetik alan verilerini ölçerek roketin oryantasyonunu ve hareketini belirler."},
        {"instruction": "UART protokolü nedir?", "input": "Haberleşme",
         "output": "Asenkron seri haberleşme protokolüdür. TX ve RX hatları üzerinden veri iletişimi sağlar."},
        {"instruction": "SPI protokolü ile I2C arasındaki fark nedir?", "input": "",
         "output": "SPI 4 hat kullanır ve daha hızlıdır. I2C 2 hat kullanır ve daha az pin tüketir."},
        {"instruction": "PWM sinyali ne için kullanılır?", "input": "Motor kontrol",
         "output": "Darbe genişlik modülasyonu ile motor hızı, servo konumu veya LED parlaklığı kontrol edilir."},
        {"instruction": "Watchdog timer nedir?", "input": "",
         "output": "Sistemin donmasını önlemek için belirli aralıklarla sıfırlanması gereken bir donanım sayacıdır."},
        {"instruction": "DMA nedir ve ne zaman kullanılır?", "input": "Bellek",
         "output": "Direct Memory Access, CPU müdahalesi olmadan bellek ile çevre birimler arasında veri aktarımı sağlar."},
        {"instruction": "GPIO pini nasıl yapılandırılır?", "input": "",
         "output": "GPIO pini input veya output olarak ayarlanır. Pull-up/pull-down dirençleri ile varsayılan durum belirlenir."},
        {"instruction": "CAN bus gömülü sistemlerde neden tercih edilir?", "input": "Otomotiv",
         "output": "Gürültüye dayanıklılığı, çok düğümlü yapısı ve güvenilir hata tespiti ile endüstriyel uygulamalarda yaygın kullanılır."},
        {"instruction": "RTOS kullanmanın avantajları nelerdir?", "input": "",
         "output": "Gerçek zamanlı görev planlaması, deterministik yanıt süresi ve çok görevli uygulama geliştirme imkânı sağlar."},
    ]
    demo_path = Path(JSONL_DIR) / 'demo.jsonl'
    with open(demo_path, 'w', encoding='utf-8') as f:
        for item in demo:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    jsonl_files = [demo_path]

for path in jsonl_files:
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    raw_data.append(json.loads(line))
                except json.JSONDecodeError:
                    pass

print(f'✅ Toplam kayıt: {len(raw_data)}')
print(f'   Örnek kayıt:')
print(f'   instruction : {raw_data[0]["instruction"]}')
print(f'   input       : {raw_data[0].get("input", "")}')
print(f'   output      : {raw_data[0]["output"][:80]}...')

In [ ]:
# Alpaca formatını Qwen chat formatına çevir
# ─────────────────────────────────────────────────────────────────────────
# Qwen2.5-Instruct şu şablonu bekler:
#   <|im_start|>system\n{sistem mesajı}<|im_end|>
#   <|im_start|>user\n{soru}<|im_end|>
#   <|im_start|>assistant\n{cevap}<|im_end|>
# ─────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = (
    'Sen TARS adlı teknik asistansın. '
    'Gömülü sistemler, roket aviyoniği ve elektronik konularında uzman bir yardımcısın. '
    'Soruları açık, teknik ve Türkçe olarak yanıtla.'
)

def format_sample(item):
    instruction = item.get('instruction', '').strip()
    inp         = item.get('input', '').strip()
    output      = item.get('output', '').strip()

    # Kullanıcı mesajını oluştur
    if inp:
        user_msg = f'{instruction}\nBağlam: {inp}'
    else:
        user_msg = instruction

    # Tam chat metni (model bunu görecek)
    text = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{user_msg}<|im_end|>\n'
        f'<|im_start|>assistant\n{output}<|im_end|>'
    )
    return {'text': text}

formatted = [format_sample(d) for d in raw_data]

print(f'✅ Formatlanan örnek sayısı: {len(formatted)}')
print('\n── İlk örnek (formatlı) ──')
print(formatted[0]['text'][:400])
print('...')

In [ ]:
from datasets import Dataset
import numpy as np

# Train / validation ayrımı  (%90 / %10)
np.random.seed(42)
indices = np.random.permutation(len(formatted))
split   = max(1, int(len(formatted) * 0.9))

train_data = [formatted[i] for i in indices[:split]]
val_data   = [formatted[i] for i in indices[split:]]

train_dataset = Dataset.from_list(train_data)
val_dataset   = Dataset.from_list(val_data)

print(f'✅ Train: {len(train_dataset)}  |  Val: {len(val_dataset)}')

## 🤖 3. Model + LoRA Kurulumu

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

print(f'Model yükleniyor: {MODEL_NAME}')
print('(İlk seferde ~1 GB indirilir, birkaç dakika sürebilir...)')

# 4-bit kuantizasyon — T4 VRAM'ini korur
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side='right',
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Model (4-bit)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config  = bnb_config,
    device_map           = 'auto',
    trust_remote_code    = True,
)
model = prepare_model_for_kbit_training(model)

print('✅ Model yüklendi')
print(f'   Toplam parametre: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# LoRA konfigürasyonu
# q_proj ve v_proj: dikkat katmanlarının sorgu ve değer matrisleri
# Bunlara LoRA uygulamak SFT için standarttır
lora_config = LoraConfig(
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    target_modules = ['q_proj', 'v_proj', 'k_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout   = LORA_DROP,
    bias           = 'none',
    task_type      = TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# VRAM kullanımı
if device == 'cuda':
    used = torch.cuda.memory_allocated() / 1e9
    print(f'   VRAM kullanımı: {used:.2f} GB')

## 🏋️ 4. Eğitim

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = 0.05,
    fp16                        = True,          # T4 fp16 destekler
    logging_steps               = 5,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    report_to                   = 'none',        # wandb kapalı
    max_seq_length              = MAX_LENGTH,
    dataset_text_field          = 'text',
    packing                     = False,
)

trainer = SFTTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    processing_class= tokenizer,
)

print('✅ Trainer hazır')
print(f'   Toplam eğitim adımı: {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM) * EPOCHS}')

In [ ]:
# Eğitimi başlat
print('🚀 Eğitim başlıyor...')
print('   (T4 üzerinde 396 kayıt × 3 epoch ≈ 10-20 dakika)')

train_result = trainer.train()

print('\n✅ Eğitim tamamlandı!')
print(f'   Train loss : {train_result.training_loss:.4f}')
print(f'   Süre       : {train_result.metrics["train_runtime"]:.0f} saniye')

In [ ]:
# LoRA adaptörünü kaydet
adapter_path = os.path.join(OUTPUT_DIR, 'final_adapter')
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'✅ Adaptör kaydedildi: {adapter_path}')

# Loss grafiği
import matplotlib.pyplot as plt

logs = [x for x in trainer.state.log_history if 'loss' in x]
train_logs = [x for x in logs if 'eval_loss' not in x and 'loss' in x]
eval_logs  = [x for x in trainer.state.log_history if 'eval_loss' in x]

fig, ax = plt.subplots(figsize=(10, 4))
if train_logs:
    ax.plot([x['step'] for x in train_logs],
            [x['loss'] for x in train_logs],
            label='Train loss', color='#0F766E', lw=2)
if eval_logs:
    ax.plot([x['step'] for x in eval_logs],
            [x['eval_loss'] for x in eval_logs],
            label='Val loss', color='#DC2626', lw=2, marker='o')
ax.set_xlabel('Adım'); ax.set_ylabel('Loss')
ax.set_title('TARS LoRA — Eğitim Kaybı', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'loss_curve.png'), dpi=150)
plt.show()
print('✅ Grafik kaydedildi')

## 💬 5. Test — TARS ile Sohbet

Eğitim bittikten sonra modelinizi burada test edin.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Adaptörü yükle (eğitimden hemen sonra çalıştırıyorsanız bu cell'i atlayabilirsiniz)
adapter_path = '/content/tars_lora/final_adapter'

print('Model yükleniyor...')
inf_tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)

inf_model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-0.5B-Instruct',
    torch_dtype   = torch.float16,
    device_map    = 'auto',
    trust_remote_code = True,
)
inf_model = PeftModel.from_pretrained(inf_model, adapter_path)
inf_model.eval()
print('✅ Model hazır')

In [ ]:
def tars_chat(soru, max_new_tokens=256, temperature=0.3):
    """
    TARS'a soru sor, cevabı döndür.
    temperature düşükse daha deterministik, yüksekse daha yaratıcı.
    """
    SYSTEM = (
        'Sen TARS adlı teknik asistansın. '
        'Gömülü sistemler, roket aviyoniği ve elektronik konularında uzman bir yardımcısın. '
        'Soruları açık, teknik ve Türkçe olarak yanıtla.'
    )
    prompt = (
        f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'
        f'<|im_start|>user\n{soru}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    inputs = inf_tokenizer(prompt, return_tensors='pt').to(inf_model.device)

    with torch.no_grad():
        outputs = inf_model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = temperature,
            do_sample      = temperature > 0,
            repetition_penalty = 1.1,
            eos_token_id   = inf_tokenizer.convert_tokens_to_ids('<|im_end|>'),
            pad_token_id   = inf_tokenizer.pad_token_id,
        )

    # Sadece yeni üretilen tokenlari al
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    answer = inf_tokenizer.decode(new_tokens, skip_special_tokens=True)
    return answer.strip()


# ── Test soruları ─────────────────────────────────────────────────────────
test_questions = [
    'ADC biriminin roket aviyonik sistemindeki rolü nedir?',
    'IMU sensörü hangi verileri ölçer?',
    'UART ile SPI arasındaki temel fark nedir?',
    'Watchdog timer neden kullanılır?',
]

print('=' * 60)
print('  TARS — Test Sonuçları')
print('=' * 60)

for q in test_questions:
    print(f'\n❓ {q}')
    answer = tars_chat(q)
    print(f'🤖 {answer}')
    print('─' * 50)

In [ ]:
# ── İnteraktif sohbet (istediğin soruyu sor) ──────────────────────────────
while True:
    soru = input('\nSorunuzu yazın (çıkmak için q): ').strip()
    if soru.lower() in ('q', 'quit', 'exit', ''):
        print('Görüşmek üzere!')
        break
    cevap = tars_chat(soru)
    print(f'\n🤖 TARS: {cevap}\n')

## 💾 6. Adaptörü Google Drive'a Kaydet (Opsiyonel)

Colab kapanırsa adaptör silinir. Drive'a kaydetmek için:

In [ ]:
# Google Drive bağla ve kaydet
from google.colab import drive
drive.mount('/content/drive')

import shutil
drive_path = '/content/drive/MyDrive/TARS_LoRA_Adapter'
shutil.copytree('/content/tars_lora/final_adapter', drive_path, dirs_exist_ok=True)
print(f'✅ Adaptör Drive\'a kaydedildi: {drive_path}')

---
## 📋 Özet

| Parametre | Değer |
|-----------|-------|
| Temel model | Qwen2.5-0.5B-Instruct |
| Kuantizasyon | 4-bit (QLoRA) |
| LoRA rank | 16 |
| LoRA alpha | 32 |
| Hedef katmanlar | q,k,v,o,gate,up,down proj |
| Epoch | 3 |
| Batch size | 4 × 4 grad. accum = 16 |
| Learning rate | 2e-4 (cosine decay) |
| Max token | 512 |

**Adaptör boyutu:** ~10-20 MB (temel model indirilmeden kullanılamaz)  
**Temel model boyutu:** ~1 GB (4-bit)

### Sonraki adımlar
- Daha fazla JSONL verisi ekleyerek yeniden eğit
- `EPOCHS=5`, `LORA_R=32` deneyin
- RAG pipeline ile birleştir (docs/ klasörünü vektör veritabanına ekle)